# Deploy and Run Llama-3.2-3B-Instruct on Modal with Authentication

This notebook deploys the Llama 3.2 3B Instruct model with a secure FastAPI HTTP endpoint and runs it with API key authentication.

### Step 1: Deploy the Authenticated Model on Modal

Run the following cell to deploy the modified hosting script. This version exposes a secure web endpoint with proxy authentication enabled.

In [1]:
# Deploy the model on Modal. The relative path points to the llm-hosting folder.
!uv run modal deploy ../llm-hosting/Llama-3-2-3B-Instruct

┌─ Error ─────────────────────────────────────────────────────────────────────┐
│ Python module paths must be specified with the -m flag. Use `modal deploy   │
│ -m ../llm-hosting/Llama-3-2-3B-Instruct` instead.                           │
└─────────────────────────────────────────────────────────────────────────────┘


### Step 2: Load Credentials from `.env` and Configure Endpoint

We load the `MODAL_KEY` and `MODAL_SECRET` (Proxy Auth tokens) from the `.env` file at the project root.

In [7]:
import os
import requests

# Load env variables from root .env file
def load_dotenv():
    # Search for .env in parent directories
    for path in [".env", "../.env", "../../.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        v = v.strip().strip("'\"")
                        os.environ.setdefault(k.strip(), v)
            break

load_dotenv()

MODAL_KEY = os.environ.get("MODAL_KEY")
MODAL_SECRET = os.environ.get("MODAL_SECRET")

if not MODAL_KEY or not MODAL_SECRET:
    print("WARNING: MODAL_KEY and MODAL_SECRET are not set. Check your .env file!")
else:
    print("Credentials loaded successfully!")

# Set your Modal workspace/profile name
username = "sshibinthomass"
ENDPOINT_URL = f"https://{username}--llama-3-2-3b-instruct-llamamodel-generate-api.modal.run"
print(f"Target Endpoint URL: {ENDPOINT_URL}")

Credentials loaded successfully!
Target Endpoint URL: https://sshibinthomass--llama-3-2-3b-instruct-llamamodel-generate-api.modal.run


### Step 3: Run Authenticated Inference

Call the secure API endpoint using the loaded credentials in HTTP headers.

In [8]:
def test_inference(prompt: str, max_new_tokens: int = 128) -> str:
    headers = {
        "Content-Type": "application/json",
        "Modal-Key": MODAL_KEY,
        "Modal-Secret": MODAL_SECRET,
    }
    payload = {
        "prompt": prompt,
        "max_new_tokens": max_new_tokens
    }
    
    print(f"Sending POST request to {ENDPOINT_URL}...")
    response = requests.post(ENDPOINT_URL, headers=headers, json=payload)
    
    if response.status_code == 200:
        # The fastapi_endpoint returns the raw string
        return response.json()
    else:
        return f"Error {response.status_code}: {response.text}"

# Run a simple test prompt
prompt = "Tell me a short joke about programming."
reply = test_inference(prompt)
print(f"\nResponse:\n{reply}")

Sending POST request to https://sshibinthomass--llama-3-2-3b-instruct-llamamodel-generate-api.modal.run...

Response:
Why do programmers prefer dark mode?

Because light attracts bugs.


### Step 4: Verify Authentication Safety

Let's test calls without headers and with invalid headers to verify that Modal's proxy authentication effectively blocks unauthorized requests.

In [9]:
print("--- Test 1: Querying without credentials ---")
try:
    response = requests.post(ENDPOINT_URL, json={"prompt": "Hello"})
    print(f"Status Code: {response.status_code} (Expected: 401)")
    print(f"Response: {response.text}")
except Exception as e:
    print("Request failed:", e)

print("\n--- Test 2: Querying with invalid credentials ---")
try:
    headers = {
        "Modal-Key": "invalid-key",
        "Modal-Secret": "invalid-secret"
    }
    response = requests.post(ENDPOINT_URL, headers=headers, json={"prompt": "Hello"})
    print(f"Status Code: {response.status_code} (Expected: 401)")
    print(f"Response: {response.text}")
except Exception as e:
    print("Request failed:", e)

--- Test 1: Querying without credentials ---
Status Code: 401 (Expected: 401)
Response: modal-http: missing credentials for proxy authorization


--- Test 2: Querying with invalid credentials ---
Status Code: 401 (Expected: 401)
Response: modal-http: invalid credentials for proxy authorization

